# CarePath DARAG — Part 1: Data Prep (CPU runtime)

Run on a **CPU runtime** (Runtime → Change runtime type → CPU). The slow step here
is real Gipformer ASR over ViMedCSS audio to build `raw_asr → gold_text` pairs
(paper §3.1) — that is CPU-bound, so running it on an L4 wastes GPU units.

Outputs saved to Google Drive (`MyDrive/carepath_artifacts`) for Part 2:
- `term_datastore.json` — NE / code-switch datastore (paper §4.2 Step 1)
- `vimedcss_gipformer_pairs.jsonl` — real GEC pairs (frozen train/val/test/hard)


In [ ]:
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer

## Get the repo into Colab

In [ ]:
# Make this CarePath repo visible to Colab. Three options:
#   1. Upload carepath.zip to /content/carepath.zip via the Files sidebar.
#   2. Set CAREPATH_REPO_ZIP to a zip path in /content or Drive.
#   3. Set CAREPATH_REPO_URL to a git URL to clone.
import os, subprocess, sys, zipfile
from pathlib import Path

REPO = Path("/content/carepath")
zip_path = os.environ.get("CAREPATH_REPO_ZIP", "/content/carepath.zip")
repo_url = os.environ.get("CAREPATH_REPO_URL")

if (REPO / "pyproject.toml").exists():
    print("Repo already present at", REPO)
elif Path(zip_path).exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content/carepath_unzip")
    # the zip may contain a top-level folder; find the dir with pyproject.toml
    roots = [p.parent for p in Path("/content/carepath_unzip").rglob("pyproject.toml")]
    src = roots[0] if roots else Path("/content/carepath_unzip")
    REPO.mkdir(exist_ok=True)
    subprocess.run(f"cp -r '{src}'/* '{REPO}'/", shell=True, check=True)
    print("Unzipped repo into", REPO)
elif repo_url:
    subprocess.run(["git", "clone", repo_url, str(REPO)], check=True)
else:
    raise SystemExit("Provide carepath.zip, CAREPATH_REPO_ZIP, or CAREPATH_REPO_URL.")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "apps" / "api"))
print("cwd:", os.getcwd())

In [ ]:
# Helper: run a pipeline CLI with PYTHONPATH set, streaming output, raising on failure.
import os, subprocess, sys

def run_step(args, env_extra=None):
    env = dict(os.environ)
    env["PYTHONPATH"] = "apps/api"
    env["PYTHONIOENCODING"] = "utf-8"
    if env_extra:
        env.update(env_extra)
    print(">>>", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed ({proc.returncode}): {' '.join(args)}")

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/carepath_artifacts")
DRIVE.mkdir(parents=True, exist_ok=True)
print("Artifacts will sync to", DRIVE)

## Run size — keep smoke defaults, raise for a real run

In [ ]:
LIMIT_PER_SPLIT = 20   # None for the full dataset
DATASET = "tensorxt/ViMedCSS"
DATASTORE = "artifacts/retrieval/term_datastore.json"
PAIRS = "artifacts/gec_pairs/vimedcss_gipformer_pairs.jsonl"
print("LIMIT_PER_SPLIT =", LIMIT_PER_SPLIT)

## 1. Build the NE / code-switch datastore (paper §4.2 Step 1)

Seeds from the curated medical lexicon and mines code-switch terms from the
dataset transcripts.

In [ ]:
run_step([
    "scripts/gec/build_datastore.py",
    "--dataset", DATASET,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--output", DATASTORE,
])

## 2. Build real Gipformer GEC pairs (CPU — the long step)

`--resume` makes this restartable if the runtime drops.

In [ ]:
run_step([
    "scripts/gec/make_pairs.py",
    "--dataset", DATASET,
    "--output", PAIRS,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--datastore", DATASTORE,
    "--resume",
])

## 3. Quick baseline WER on the raw ASR (paper Table 1 style)

In [ ]:
run_step([
    "scripts/gec/evaluate.py",
    "--input", PAIRS,
    "--prediction-columns", "raw_asr",
    "--wer-output", "artifacts/evaluations/raw_baseline_wer.json",
    "--ne-f1-output", "artifacts/evaluations/raw_baseline_ne_f1.json",
])

## 4. Save artifacts to Google Drive (for Part 2)

In [ ]:
import shutil
from pathlib import Path
for rel in [DATASTORE, PAIRS, "artifacts/evaluations/raw_baseline_wer.json"]:
    src = Path(rel)
    if src.exists():
        dst = DRIVE / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print("saved", dst)
print("Part 1 done — switch to Part 2 on a GPU runtime.")